# AccentMark: Step 1 - Phoneme-Level Feature Extraction

This notebook implements the feature extraction pipeline for the **AccentMark** project. 
It reads audio files and their force-aligned phoneme boundaries from TextGrids, extracts acoustic feature vectors for each phoneme occurrence, and computes speaker-level phoneme means.

To facilitate speaker re-identification analysis in the next notebook, we extract:
1. **Full session means**: Averaged across all available utterances for the speaker (used for primary modeling and visualization).
2. **Split session means**: Utterances are split into two halves (e.g., `_split1` and `_split2`) to act as separate sessions for verification.

## 1. Imports and Workspace Configuration

In [ ]:
import os
import pickle
import sys

# Ensure the src directory is in the python path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.utils import get_speaker_files
from src.features import extract_speaker_phoneme_means
from src.generate_synthetic_data import generate_all_synthetic_data

## 2. Define Speakers and Dataset Directories

We define the 14 speakers in the L2-ARCTIC corpus representing 6 L1 backgrounds and native English speakers.

In [ ]:
SPEAKERS = {
    'ABA': 'Arabic', 'YBAA': 'Arabic',
    'SVBI': 'Hindi', 'TNI': 'Hindi',
    'HJK': 'Korean', 'YDCK': 'Korean',
    'BWC': 'Mandarin', 'LXC': 'Mandarin',
    'EBVS': 'Spanish', 'ERMS': 'Spanish',
    'HQTV': 'Vietnamese', 'PNV': 'Vietnamese',
    'NJS': 'Native', 'TXHC': 'Native'
}

DATA_DIR = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Dataset Check & Synthetic Data Fallback

If the L2-ARCTIC dataset is not found in `data/` or contains fewer than 30 files per speaker, we generate a synthetic mock dataset to allow the code to run end-to-end immediately.

In [ ]:
dataset_missing = False
for speaker in SPEAKERS.keys():
    spk_wav_path = os.path.join(DATA_DIR, speaker, 'wav')
    if not os.path.exists(spk_wav_path) or len(os.listdir(spk_wav_path)) < 30:
        dataset_missing = True
        break

if dataset_missing:
    print("L2-ARCTIC dataset not found, incomplete, or synthetic data needs regeneration in data/.")
    print("Triggering synthetic data generator to create 40 utterances per speaker...")
    # Generate 40 utterances per speaker to ensure splits have >= 5 occurrences per phoneme
    generate_all_synthetic_data(DATA_DIR, num_utterances=40)
else:
    print("Dataset files verified. Starting pipeline with real or cached L2-ARCTIC dataset.")

## 4. Run Feature Extraction

We iterate over all speakers and compute their full-session phoneme means and split-half phoneme means (first half and second half of utterances).

In [ ]:
all_means = {}

for speaker_id, l1 in SPEAKERS.items():
    print(f"==================================================")
    print(f"Extracting features for Speaker: {speaker_id} ({l1})")
    print(f"==================================================")
    
    # Get list of matched audio/annotation files
    speaker_files = get_speaker_files(DATA_DIR, speaker_id)
    total_files = len(speaker_files)
    print(f"Found {total_files} utterances.")
    
    if total_files == 0:
        print(f"Skipping speaker {speaker_id} due to no files.")
        continue
        
    # 1. Full session phoneme extraction
    print("--- Extracting Full Session ---")
    full_means = extract_speaker_phoneme_means(speaker_files, max_utterances=150)
    all_means[speaker_id] = full_means
    print(f"Full session: Retained {len(full_means)} phonemes with >= 5 occurrences.")
    
    # 2. Split session phoneme extraction for verification metrics
    mid_point = total_files // 2
    files_split1 = speaker_files[:mid_point]
    files_split2 = speaker_files[mid_point:]
    
    print("--- Extracting Split 1 (First Half) ---")
    means_split1 = extract_speaker_phoneme_means(files_split1, max_utterances=75)
    all_means[f"{speaker_id}_split1"] = means_split1
    
    print("--- Extracting Split 2 (Second Half) ---")
    means_split2 = extract_speaker_phoneme_means(files_split2, max_utterances=75)
    all_means[f"{speaker_id}_split2"] = means_split2
    
    print(f"Splits completed: Split 1 has {len(means_split1)} phonemes, Split 2 has {len(means_split2)} phonemes.\n")

## 5. Save Features to Disk

In [ ]:
output_pkl = os.path.join(RESULTS_DIR, 'all_means.pkl')
with open(output_pkl, 'wb') as f:
    pickle.dump(all_means, f)

print(f"Successfully saved phoneme means for {len(all_means)} speaker configurations to {output_pkl}")